<a href="https://colab.research.google.com/github/salekmasudparvez1/deep-learning-note/blob/main/nlp/text_prepros/text_prepro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [10]:
import pandas as pd
from symspellpy import SymSpell, Verbosity

from pathlib import Path
import nltk
import re
import string
import os
import requests
import zipfile



In [11]:
dataset_dir = Path.cwd()/"content/dataset"
os.makedirs(dataset_dir,exist_ok=True)

In [12]:
destination_path = os.path.join(dataset_dir,"movie_data.zip")

url = "https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"

response = requests.get(url,stream=True)

with open(destination_path, 'wb') as f :
  for chunk in response.iter_content(chunk_size=8192):
    f.write(chunk)

print("Download complete!")
with zipfile.ZipFile(destination_path,'r') as z:
  z.extractall(dataset_dir)
  print(f"Successfully extracted all files to: {dataset_dir}")
print("Files in directory:", os.listdir(dataset_dir))


Download complete!
Successfully extracted all files to: /home/salekmasudparvez/coding_ai_ml/deep-learning-note/nlp/text_prepros/content/dataset
Files in directory: ['movie_data.zip', 'IMDB Dataset.csv']


In [13]:
data_frame_path = Path.cwd()/"content/dataset/IMDB Dataset.csv"
df = pd.read_csv(data_frame_path)
df.sample(5)
# df = pd.read_csv(
#     "/content/IMDB Dataset.csv",
#     encoding="latin1",
#     engine="python",
#     on_bad_lines="skip"
# )


,review,sentiment
7768,Saw this late one night on cable. At the time ...,negative
10743,This is the best mob film ever made. It deserv...,positive
19157,"Well, shuck me sideways. I haven't seen a home...",negative
29547,I saw this movie after i saw Blue Crush and ot...,positive
18377,After watching the series premiere of Talk Sho...,positive


In [14]:
(df.columns)

Index(['review', 'sentiment'], dtype='str')

In [15]:
#  make all th review lower

df['review'] = df['review'].str.lower()


TAG_PATTERN = re.compile("<.*?>")
URL_PATTERN = re.compile("https?://\S+|www\.\S+")
EMOJIES_PATTERN = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # Emoticons 😀 😃 😂
        "\U0001F300-\U0001F5FF"  # Symbols and pictographs 🌍 🎉
        "\U0001F680-\U0001F6FF"  # Transport and maps 🚀 🚗
        "\U0001F1E0-\U0001F1FF"  # Flags 🇧🇩 🇺🇸
        "\U0001F900-\U0001F9FF"  # Supplemental symbols 🤖 🦾
        "\U0001FA70-\U0001FAFF"  # Extended pictographs 🫠 🪿
        "\U00002702-\U000027B0"  # Dingbats ✂️ ⭐
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
# URL_PATTERN = re.compile(
#     r'(?i)\b(?:https?://)?'
#     r'(?:www\.)?'
#     r'(?:localhost|(?:\d{1,3}\.){3}\d{1,3}|'
#     r'(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+'
#     r'[a-z]{2,63})'
#     r'(?::\d{1,5})?'
#     r'(?:[/?#][^\s<>"\']*)?'
# )
# remove tags
punctuation = string.punctuation

def remove_tag (row):
  return TAG_PATTERN.sub("",row)

def remove_url(text):
  return URL_PATTERN.sub(r"",text)

def remove_puntuation(row):

  for char in punctuation:
    row = row.replace(char,"")
    return row

def remove_puntuation2(text):
  return text.translate(str.maketrans(' ',' ',punctuation))

def remove_emojies (row):
  return EMOJIES_PATTERN.sub("",row)


<>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
/tmp/ipykernel_125193/2848146121.py:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  URL_PATTERN = re.compile("https?://\S+|www\.\S+")


In [16]:
df['review'] = df['review'].apply(remove_tag)
df['review'] = df['review'].apply(remove_url)
df['review'] = df['review'].apply(remove_puntuation)
df['review'] = df['review'].apply(remove_emojies)

In [17]:
chat_words = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can’t Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don’t Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn’t Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired",
    "OMG!": "Oh My God",
    "OMG":"Oh My God",
    "SMH": "Shaking My Head",
    "NGL": "Not Gonna Lie",
    "TBF": "To Be Fair",
    "ICYMI": "In Case You Missed It",
    "IMO": "In My Opinion",
    "IMHO": "In My Honest Opinion",
    "RN": "Right Now",
    "FR": "For Real",
    "DW": "Don't Worry",
    "ILYSM": "I Love You So Much",
    "OMW": "On My Way",
    "GGWP": "Good Game Well Played",
    "GG": "Good Game",
    "EZ": "Easy",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud"
}

In [18]:
def chat_conversion (text):
  new_text = []
  for w in text.split():
    print(w.upper() in chat_words)
    if w.upper() in chat_words:
      print(w)
      new_text.append(chat_words[w.upper()])
    else :
      new_text.append(w)
  return " ".join(new_text)
chat_conversion("OMG! fyp")

True
OMG!
True
fyp


'Oh My God For You Page'

# **Spell checking**


In [19]:
DICTIONARY_URL = "https://raw.githubusercontent.com/wolfgarbe/SymSpell/refs/heads/master/SymSpell/frequency_dictionary_en_82_765.txt"
BIOGRAM_DICTIONARY_URL = "https://raw.githubusercontent.com/wolfgarbe/SymSpell/refs/heads/master/SymSpell/frequency_bigramdictionary_en_243_342.txt"

DICTIONARY_DIR = Path.cwd()/"content/dictionary"

os.makedirs(DICTIONARY_DIR, exist_ok=True)

WORD_DICTIONARY_DIR = os.path.join(DICTIONARY_DIR, "dictionary.txt")
BIOGRAM_DICTIONARY_DIR = os.path.join(DICTIONARY_DIR, "biogram_dictionary.txt")

# Correct syntax: !curl -fL <URL> -o <local_destination_path>
!curl -fL {DICTIONARY_URL} -o {WORD_DICTIONARY_DIR}
!curl -fL {BIOGRAM_DICTIONARY_URL} -o {BIOGRAM_DICTIONARY_DIR}

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  1.27M 100  1.27M   0      0  1.95M      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  4.89M 100  4.89M   0      0  2.25M      0   00:02   00:02          2.20M


In [20]:

sym_spell = SymSpell(
    max_dictionary_edit_distance=2,
    prefix_length=7
)

sym_spell.load_dictionary(
    WORD_DICTIONARY_DIR,
    term_index=0,
    count_index=1
)
sym_spell.load_bigram_dictionary(
    BIOGRAM_DICTIONARY_DIR,
    term_index=0,
    count_index=2
)
print("Dictionary loaded successfully!")

Dictionary loaded successfully!


In [21]:
text = "the quik brown fox jumps over teh lazy dog"

suggestions = sym_spell.lookup_compound(
    text,
    max_edit_distance=2
)
for i in suggestions:
  print(i.term)


the quick brown fox jumps over tech lazy dog


# **Stopwords removal**


In [22]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/salekmasudparvez/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [23]:
stop_word = nltk.corpus.stopwords.words('english')
stop_word

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [24]:
def remove_stopword (row_text):
  split_text = row_text.split()
  clean_text =[]
  for word in split_text:
    if word and word not in stop_word:
      clean_text.append(word)
  return " ".join(clean_text)

In [25]:
df['review'] = df['review'].apply(remove_stopword)

# **Tokenization**


In [26]:
from nltk import word_tokenize, sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/salekmasudparvez/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/salekmasudparvez/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [27]:
sent5 = 'I have a Ph.D in A.I'
sent6 = "We're here to help! mail us at nks@gmail.com"
sent7 = 'A 5km ride cost $10.50'

print(word_tokenize(sent7))
print(sent_tokenize(sent7))

['A', '5km', 'ride', 'cost', '$', '10.50']
['A 5km ride cost $10.50']


# **Spacy**


In [45]:
!pip install -U spacy

  Using cached spacy-3.8.2.tar.gz (1.3 MB)
  Installing build dependencies ... \anceled
ERROR: Operation cancelled by user
^C


In [48]:
import spacy

ModuleNotFoundError: No module named 'spacy'

In [ ]:
nlp = spacy.load('en_core_web_lg')
doc = nlp(sent6)


# **stemming**


In [ ]:
from nltk.stem.porter import PorterStemmer

In [ ]:
ps = PorterStemmer()
def stemming_word(text):
  result = []
  for word in text.split():
    result.append(ps.stem(word))
  return " ".join(result)

def stem_words(text):
    return " ".join([ps.stem(word) for word in text.split()])

In [ ]:
stemming_word('running run')

'run run'

# Lemmatization


In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/salekmasudparvez/nltk_data...


True

In [ ]:
punctuations=string.punctuation

sentence = "He was running? and eating at same time. He has bad habit of swimming after playing long hours in the Sun."
wordnet_lemmatizer = WordNetLemmatizer()
sentence_words = nltk.word_tokenize(sentence)

In [ ]:
for word in sentence_words:
  if word in punctuation:
    print(word)
    sentence_words.remove(word)


?
.
.
